# GroundedAI: LoRA fine-tuning of the answer model on a free Colab GPU

This notebook fine-tunes **Qwen/Qwen2.5-1.5B-Instruct** (the model the GroundedAI demo uses) with
LoRA, then compares it with the original model through GroundedAI's own evaluation pipeline.

**It uses only the free tier.** Do not buy Colab Pro or compute units for it. If no GPU is
available, stop here and try again later; the notebook does not try to work around Colab's limits.

**Before you start:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU* (or any GPU
offered for free), then *Runtime → Run all*. The first cell stops the notebook if there is no GPU.

| Step | What happens | Time on a T4 (estimate) |
|---|---|---|
| 1 | hardware report | seconds |
| 2-4 | settings, Google Drive, code and data | 3-5 min |
| 5 | smoke test: 3 training steps at 512 tokens | 2-3 min |
| 6 | training (resumable from Google Drive) | about 1 hour |
| 7 | the adapter loads on the base model; one example answer | 1-2 min |
| 8 | base vs fine-tuned on the validation sets | 20-40 min |
| 9 | base vs fine-tuned on the protected test set, **once** | 10-20 min |
| 10 | merge check, results archive | 5 min |

The times are estimates, not measurements. Everything the notebook writes goes to
`MyDrive/groundedai-lora/<run name>/` when Google Drive is used, so a disconnect loses at most the
steps since the last checkpoint: run the notebook again with the same run name and training continues.

Details, data and results: [docs/FINETUNING.md](https://github.com/pouyapd/GroundedAI/blob/main/docs/FINETUNING.md).

## 1. Hardware report

In [ ]:
import json, shutil, subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi: no GPU visible")
import torch, psutil
report = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "ram_gb": round(psutil.virtual_memory().total / 2**30, 1),
    "disk_free_gb": round(shutil.disk_usage("/content").free / 2**30, 1),
}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    report.update(gpu=props.name, gpu_memory_gb=round(props.total_memory / 2**30, 1),
                  compute_capability=f"{props.major}.{props.minor}", bf16_supported=torch.cuda.is_bf16_supported())
print(json.dumps(report, indent=2))
if not torch.cuda.is_available():
    raise SystemExit("No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, or try again later. Nothing was trained.")
gb = report["gpu_memory_gb"]
print("Verdict:", "suitable for Qwen2.5-1.5B fp16 LoRA" if gb >= 14 else
      "probably suitable; the sequence length may be reduced automatically" if gb >= 10 else
      "too little memory for the 1.5B model: set BASE_MODEL to Qwen/Qwen2.5-0.5B-Instruct below")
if sys.version_info < (3, 12):
    raise SystemExit("GroundedAI needs Python 3.12 or later; this runtime has " + sys.version.split()[0])

## 2. Settings

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_NAME = "run1"            # keep it to resume; change it for a new run
USE_DRIVE = True             # keep checkpoints and results in Google Drive
REPO = "https://github.com/pouyapd/GroundedAI.git"
REPO_REF = "main"            # a branch, tag or commit; the commit used is recorded
EPOCHS = 2
LEARNING_RATE = 1e-4
LORA_RANK = 16

## 3. Google Drive (for checkpoints and results)

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/groundedai-lora/{RUN_NAME}"
else:
    OUT = f"/content/groundedai-lora/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
print("Results go to", OUT)

## 4. Code, libraries and data

In [ ]:
%cd /content
!rm -rf GroundedAI && git clone --quiet {REPO} GroundedAI && cd GroundedAI && git checkout --quiet {REPO_REF} && git log -1 --oneline
%cd /content/GroundedAI
!pip install --quiet -r training/requirements-colab.txt
!pip install --quiet -e backend
import hashlib, json
expected = json.load(open("training/data/dataset_report.json"))["sha256"]
for name, digest in expected.items():
    actual = hashlib.sha256(open(f"training/data/{name}", "rb").read()).hexdigest()
    assert actual == digest, f"{name} differs from dataset_report.json"
print("training data verified:", json.load(open("training/data/dataset_report.json"))["totals"])

## 5. Smoke test (3 steps at 512 tokens)

Prompts are cut to 512 tokens here, so this only checks that training runs on this GPU.

In [ ]:
!python training/finetune_lora.py --base {BASE_MODEL} --out /content/smoke --smoke

## 6. Training

The sequence length is chosen by trying the longest example at 2048, 1536 and 1024 tokens on this
GPU. If the runtime disconnects, run the notebook again with the same `RUN_NAME`: training
continues from the latest checkpoint in Drive. The best checkpoint by validation loss is kept.

In [ ]:
!python training/finetune_lora.py --base {BASE_MODEL} --out {OUT} --epochs {EPOCHS} --lr {LEARNING_RATE} --rank {LORA_RANK}
print(open(f"{OUT}/result.json").read())

## 7. Load the adapter on the base model, and one example

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(f"{OUT}/tokenizer")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16).to("cuda").eval()
row = [json.loads(l) for l in open("training/data/validation.jsonl")][0]
prompt = tok.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
def answer(model):
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("QUESTION:", row["question"]); print("TARGET:  ", row["target"])
print("BASE:    ", answer(base))
tuned = PeftModel.from_pretrained(base, f"{OUT}/adapter").eval()
print("LORA:    ", answer(tuned))
del base, tuned; torch.cuda.empty_cache()

## 8. Validation: base vs fine-tuned

Same index, retrieval, prompt, checks and greedy decoding for both. The held-out fictional
organisations and the NIST validation documents were not trained on.

In [ ]:
!python scripts/fetch_nist_corpus.py > /dev/null && echo "NIST PDFs downloaded and hash-checked"
import subprocess
REPORTS = f"{OUT}/reports"
os.makedirs(REPORTS, exist_ok=True)
VALIDATION = {"synthetic-eval": "training/data/synthetic-eval",
              "nist-val": "evaluation/datasets/groundedai-nist-val-v1",
              "nist-val-relevance": "evaluation/datasets/groundedai-nist-val-relevance-v1"}
def evaluate(name, path, label, adapter=None):
    target = f"{REPORTS}/{name}-{label}.json"
    if os.path.exists(target):
        print("already done:", target); return
    cmd = ["python", "training/evaluate_finetuned.py", "--base", BASE_MODEL, "--dataset", path, "--label", label, "--out", target]
    if adapter:
        cmd += ["--adapter", adapter]
    subprocess.run(cmd, check=True)
for name, path in VALIDATION.items():
    evaluate(name, path, "base")
    evaluate(name, path, "lora", f"{OUT}/adapter")
    subprocess.run(["python", "scripts/compare_rag_reports.py", f"{REPORTS}/{name}-base.json", f"{REPORTS}/{name}-lora.json",
                    "--out", f"{REPORTS}/{name}-comparison.json"], check=True, stdout=subprocess.DEVNULL)
    print(name, json.dumps(json.load(open(f"{REPORTS}/{name}-comparison.json"))["measures"], indent=1))

## 9. Protected test set: run once

`groundedai-nist-v1` (53 questions) was not used for training or for choosing anything. Run this
cell **once**, after the adapter is final; do not change the training because of its result. If
it has already run for this run name, it is not run again.

In [ ]:
TEST = "evaluation/datasets/groundedai-nist-v1"
evaluate("test", TEST, "base")
evaluate("test", TEST, "lora", f"{OUT}/adapter")
!python scripts/compare_rag_reports.py {REPORTS}/test-base.json {REPORTS}/test-lora.json --out {REPORTS}/test-comparison.json > /dev/null
print(json.dumps(json.load(open(f"{REPORTS}/test-comparison.json"))["measures"], indent=1))

## 10. Merge check and results archive

The merged model must answer like the adapter on a few questions. The archive holds the reports, logs and the adapter (tens of MB), not the merged model.

In [ ]:
subprocess.run(["python", "training/evaluate_finetuned.py", "--base", BASE_MODEL, "--dataset", VALIDATION["synthetic-eval"],
                "--label", "lora-merged", "--adapter", f"{OUT}/adapter", "--merge", "--limit", "20",
                "--out", f"{REPORTS}/merge-check-merged.json"], check=True)
a = {r["question_id"]: r.get("model_output") for r in json.load(open(f"{REPORTS}/synthetic-eval-lora.json"))["per_question"]}
b = {r["question_id"]: r.get("model_output") for r in json.load(open(f"{REPORTS}/merge-check-merged.json"))["per_question"]}
same = sum(a[q] == b[q] for q in b)
print(f"merged model: same answer as the adapter on {same} of {len(b)} questions")
json.dump({"same": same, "of": len(b)}, open(f"{REPORTS}/merge-check.json", "w"))
!git rev-parse HEAD > {OUT}/groundedai_commit.txt
!cd {OUT} && zip -qr /content/groundedai-lora-results.zip adapter tokenizer reports *.json groundedai_commit.txt
print("Download /content/groundedai-lora-results.zip (Files panel) or find everything in", OUT)

## Browser (WebLLM) deployment: not done here

The demo runs Qwen2.5-1.5B through WebLLM as an MLC 4-bit build (`q4f16_1`). The fine-tuned model
has the same architecture, so in principle the merged weights can be converted with MLC LLM
(`mlc_llm convert_weight --quantization q4f16_1` and `mlc_llm gen_config`) and served with the
existing Qwen2.5-1.5B WebGPU library. That needs the MLC LLM package, a place to host the
converted weights (for example a Hugging Face model repository) and a change to the demo; none of
it has been done or tested. See docs/FINETUNING.md.